# C3 exp11, the two kNN runs: each segment's own examples, chosen by similarity (KATE)

The first exp11 notebook (`notebooks/kaggle_c3_exp11.ipynb`, commit `b992af6`) finished its **random** run (macro-F1 0.325). Both kNN runs stopped before the 7B was called: `nearest_per_class` indexed the class rows (on the CPU) with `topk`'s positions (on the GPU). That is fixed (`src/smi/prompting/retrieve.py`), and step 5 below checks it on the GPU before anything long starts. This notebook runs only the two kNN runs, with the settings of the first notebook:
- **knn_tuned** (GPU 0): DeBERTa-v3-base fine-tuned on train with C1's code and adopted setting (lr 1e-5, 3 epochs, seed 42, last epoch kept), `[CLS]` vector. Then the 7B with each segment's 2 closest train examples of every label.
- **knn_base** (GPU 1): DeBERTa-v3-base as released, mean of the last layer. Then the 7B the same way.

Also written, without the 7B: `shots/<name>/nearest/` (each segment labelled with its closest example's label: the retriever alone) and `shots/knn_tuned/encoder/` (the fine-tuned encoder's own predictions).

Speed: `--prefix-cache`, batch 16, as the random run. Only a run stopped by its cache check runs again without the cache, as `<run>_nocache`.

Note: `tasks/C3-prompting-baseline/experiments/exp11_knn-shots-7b/NOTE.md`. Temporary trials (`scripts/trial_prompting.py`); not reportable (PROTOCOL §4.1).

Before running, in the notebook settings: **GPU T4 x2**, **Internet on**, the `smi-v3-meetings` dataset attached (only that one: the data cell looks for v3).

Expected time: about 2 to 2.5 hours. The fine-tune and embeddings take ~20 min (knn_tuned) or ~3 min (knn_base). Each 7B run then takes ~100 min: the random run scored 2.8 segments/s.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
`bitsandbytes` for the 4-bit 7B, `sentencepiece` for DeBERTa's tokenizer (as in C1's notebooks). Do **not** install `requirements.txt` here.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 bitsandbytes sentencepiece==0.2.2

## 3. Data folder: the v3 one

In [ ]:
import glob, json, os, subprocess, time

hits = [h for h in glob.glob("/kaggle/input/**/source.json", recursive=True) if json.load(open(h))["data_version"] == "v3"]
assert len(hits) == 1, f"expected exactly one v3 dataset under /kaggle/input, found {hits} : is smi-v3-meetings attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "|", json.load(open(hits[0])))
os.makedirs("/kaggle/working/logs", exist_ok=True)

## 4. Download the models once

In [ ]:
from huggingface_hub import snapshot_download

SEVEN_B = "Qwen/Qwen2.5-7B-Instruct"
ENCODER = "microsoft/deberta-v3-base"
for name in (SEVEN_B, ENCODER):
    print(snapshot_download(name))

## 5. Check the retrieval on the GPU
The step that stopped the first notebook, on a tiny case with a known answer. If this cell fails, stop here.

In [ ]:
CHECK = """
import torch
from smi.prompting.retrieve import nearest_per_class

pool, labels = [], []
for c in range(7):
    for toward_query in (0.5, 2.0, 1.0):  # per label : the second row is the closest , then the third
        row = [0.0] * 8
        row[c], row[7] = 1.0, toward_query
        pool.append(row)
        labels.append(c)
pool = torch.nn.functional.normalize(torch.tensor(pool), dim=-1).cuda()
rows, _ = nearest_per_class(torch.tensor([[0.0] * 7 + [1.0]]).cuda(), pool, labels, per_class=2)
assert rows == [[3 * c + r for c in range(7) for r in (1, 2)]], rows
print("nearest_per_class on the GPU: OK")
"""
subprocess.run(["python", "-c", CHECK], cwd="/kaggle/working/repo", check=True,
               env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONPATH="src"))

## 6. Helpers: shots files, runs, a queue per GPU, and waiting while showing progress
Every step logs to `logs/`. A shots file that already exists is not made again; a half-written one (folder without `shots.jsonl`) must be deleted by hand first. A run starts only if its shots file was made.

In [ ]:
SHOTS = "/kaggle/working/shots"
RETRIEVE = {
    "knn_tuned": f"--select similar --encoder {ENCODER} --fine-tune --seed 42 --pooling cls",
    "knn_base": f"--select similar --encoder {ENCODER} --pooling mean",
}

def retrieve(name):
    """make the shots file SHOTS/<name> , unless it is already there"""
    return (f"[ -f {SHOTS}/{name}/shots.jsonl ] || python scripts/retrieve_shots.py --data-root '{DATA_ROOT}' "
            f"--out {SHOTS}/{name} --per-class 2 {RETRIEVE[name]} > /kaggle/working/logs/shots_{name}.log 2>&1")

def score(name):
    """the 7B with each segment's examples from SHOTS/<name> ; only if the prefix-cache check stops it , again without the cache"""
    run = f"c3_exp11_7b_{name}"
    log = f"/kaggle/working/logs/{run}.log"
    command = (f"python scripts/trial_prompting.py --data-root '{DATA_ROOT}' --out-dir /kaggle/working/runs "
               f"--model {SEVEN_B} --load-in-4bit --shots-file {SHOTS}/{name} --batch-size 16")
    return (f"{command} --run-name {run} --prefix-cache > {log} 2>&1 || "
            f"( grep -q 'further from full prompts than allowed' {log} && "
            f"{command} --run-name {run}_nocache > /kaggle/working/logs/{run}_nocache.log 2>&1 )")

def arm(name):
    """the shots file , then (only if it was made) the 7B run"""
    return f"( {retrieve(name)} ) && ( {score(name)} )"

def queue(gpu, command):
    """run the command on one GPU , in the background"""
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src", TOKENIZERS_PARALLELISM="false")
    return subprocess.Popen(command, shell=True, cwd="/kaggle/working/repo", env=env)

def last_line(path):
    lines = [l for l in open(path).read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

def wait(processes):
    """every 2 minutes , print the latest line of every log , until all queues have finished"""
    while any(p.poll() is None for p in processes):
        time.sleep(120)
        for path in sorted(glob.glob("/kaggle/working/logs/*.log")):
            print(f"{os.path.basename(path)[:-4]:30} {last_line(path)[-80:]}")
        print()
    print("all queues finished")

## 7. Run both GPUs (~2 to 2.5 h)
- GPU 0: fine-tune the encoder and make `knn_tuned` (~20 min), then its 7B run.
- GPU 1: make `knn_base` (~3 min), then its 7B run.

If the session stops, run this cell again: finished steps are skipped and an interrupted 7B run continues where it stopped.

In [ ]:
wait([queue(0, arm("knn_tuned")), queue(1, arm("knn_base"))])

## 8. Results
From the first notebook (v3): per-segment random examples 0.325. Baselines: exp08's one fixed draw of 2 examples per label, 0.341; zero-shot 0.206. Prompt noise (exp07): differences below ~0.035 are not real. The fine-tuned encoder alone scored 0.522 in the first notebook.

In [ ]:
def show(name, path):
    if not os.path.exists(f"{path}/metrics.json"):
        print(f"{name:32} (missing : see logs/)"); return
    m = json.load(open(f"{path}/metrics.json"))
    predicted_other = sum(json.loads(line)["label"] == "other" for line in open(f"{path}/predictions.jsonl"))
    per_class = " ".join(f"{label} {m['per_class'][label]['f1']:.3f}" for label in ("cs", "co", "aa", "bk", "ar", "cc"))
    print(f"{name:32} {m['primary']['macro_f1_6']:8.4f} {predicted_other:11,}   {per_class}")

print(f"{'':32} {'macro-F1':>8} {'pred. other':>11}   (gold other 13,634)")
for name in RETRIEVE:
    run = f"c3_exp11_7b_{name}"
    if not os.path.exists(f"/kaggle/working/runs/{run}/metrics.json"):
        run += "_nocache"
    show(run, f"/kaggle/working/runs/{run}")
for name in RETRIEVE:
    show(f"{name} : retriever alone", f"{SHOTS}/{name}/nearest")
show("knn_tuned : encoder alone", f"{SHOTS}/knn_tuned/encoder")

## 9. Download the outputs
Right-hand panel: **Output** → `/kaggle/working` → `c3_exp11_knn_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_exp11_knn_runs.zip runs shots logs
!ls -la c3_exp11_knn_runs.zip